# braga2025annotation

> Semi-automatic annotation process of AIRCloud.

In [ ]:
#| eval: false
from torchvision.transforms import v2
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from colorcloud.behley2019iccv import SemanticKITTIDataset, SphericalProjection
from colorcloud.behley2019iccv import ProjectionTransform, ProjectionToTensorTransform
import os
import yaml
import shutil
import zipfile

## Introduction

This module aims to simplify and accelerate the process of annotating 3D point clouds through a semi-automatic approach. The methodology adopted is based on using a semantic segmentation model as the initial tool for automatic annotation. This model performs inference on the point cloud, generating preliminary labels for the various elements present in the scene.

Subsequently, a human verifier reviews these annotations using a specific annotation tool, correcting any errors made by the model. After this stage, the validated annotations are incorporated into the dataset, expanding it and enabling the training of a new segmentation model, thus progressively improving the quality of the annotations generated in each iteration.

It is important to emphasize that the human verifier corrects only the new annotations produced by the model. Previously reviewed annotations are marked with a distinct label during the verification process, helping to identify new points.

## Setup

The first step is to define the folder paths. It is important to note that the script currently expects only one dataset sequence at a time.

- `DATASET_PATH`: Path containing the dataset
- `OUTPUT_PATH`: Location where the ZIP file with the labels will be saved
- `SEQUENCE` : Selected sequence

In [ ]:
#| eval: false
DATASET_PATH = "/workspace/data/AIRCloud_V3_F/"
OUTPUT_PATH = "/workspace/converted_labels/"
SEQUENCE = "08"

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

## Unifying Dataset Labels
::: {.callout-important}
If this is the first iteration of the dataset in the semi-automatic annotation process, skip this step.
:::

After completing the corrections, the verified labels must be reverted to their original information and the dataset unified.  

To do this, add the compressed folder containing the labels to `/workspace/data/` and set its path in `ZIP_PATH`. Also, define the output path for the new labels.

In [ ]:
#| eval: false
ZIP_PATH = "/workspace/data/08_teste.zip"
CORRECTED_DATA_PATH = '/workspace/data/08_teste'

def unzip_file(zip_file_path, destination):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(destination)
        print(f"Files extracted to: {destination}")

os.makedirs(CORRECTED_DATA_PATH, exist_ok=True)
unzip_file(ZIP_PATH, CORRECTED_DATA_PATH)

Files extracted to: /workspace/data/08_teste


Loads the newly annotated and verified labels together with the previous labels in order to restore the original information and create the unified dataset.

`new_labels_path` : path to the new corrected labels
`old_labels_path` : 

In [ ]:
#| eval: false
def load_labels(labels_path, label_list):
    labels = []
    for index in (range(len(label_list))):
        label_path = os.path.join(labels_path, label_list[index])
        with open(label_path, 'rb') as f:
            label = np.fromfile(f, dtype=np.uint32)
            label = label & 0xFFFF
        labels.append(label)

    return labels

new_labels_path = os.path.join(CORRECTED_DATA_PATH, "labels")
new_labels_list = sorted(os.listdir(new_labels_path))
new_labels = load_labels(new_labels_path, new_labels_list)

old_labels_path = os.path.join(DATASET_PATH, "data_odometry_labels", "dataset", "sequences", SEQUENCE, "labels")
old_labels_list = sorted(os.listdir(old_labels_path))
old_labels = load_labels(old_labels_path, old_labels_list)

**Mapping rule:** For each element, use the value from `new_labels` unless it is a reserved/placeholder class (`0` or `99`). Otherwise, keep the value from `old_labels`. The result is stored in `mapped_labels`.

In [ ]:
#| eval: false
mapped_labels = []
for old, new in zip(old_labels, new_labels):
    mask = (new != 99) & (new != 0)
    mapped = np.where(mask, new, old)
    mapped_labels.append(mapped)

Create a temporary output folder, write each mapped array as a binary file (`uint32`), and then compress the folder into a `.zip` archive for download.

In [ ]:
#| eval: false
folder = "/workspace/converted_labels/temp_labels/"
os.makedirs(folder, exist_ok=True)

for filename, arr in zip(label_list, mapped_labels):
    final_labels_uint32 = arr.astype(np.uint32)
    out_path = os.path.join(folder, filename)
    with open(out_path, "wb") as f:
        final_labels_uint32.tofile(f)

zip_filename = "/workspace/converted_labels/08_teste_new"
shutil.make_archive(zip_filename, "zip", folder)
shutil.rmtree(folder)   

print(f"Labels ready for download at {zip_filename}.zip")

Labels ready for download at /workspace/converted_labels/08_teste_new.zip


## Inference

In this example, the network used will be **RIUNet**, which takes as input point clouds previously processed into range images. Since **AIRCloud** was built with a structure very similar to **SemanticKITTI**, we can use the `00_behley2019iccv` module to process and load the dataset, simply by adjusting the projection parameters and passing the `aircloud` flag to the class. For more information on how to load the dataset, refer to the `00_behley2019iccv` notebook.

::: {.callout-important}
The `SemanticKITTIDataset` class from the `00_behley2019iccv` module, although it fully supports loading **AIRCloud**, does not yet provide ways to specify which  `.yaml` file to use or which sequences to load. To work around this, it is important to modify the `.yaml` in the Splits section: set the desired sequence under the `valid` key and use the parameter `split="valid"`. This way, only the target sequence will be loaded.
:::

In [ ]:
#| eval: false
proj = SphericalProjection(fov_up_deg=15., fov_down_deg=-15., W=1024, H=16)
tfms = v2.Compose([
    ProjectionTransform(proj),
    ProjectionToTensorTransform(),
])

ds = SemanticKITTIDataset(
    data_path=DATASET_PATH, 
    split='valid', 
    transform=tfms, 
    aircloud=True
)

print("Size of the dataset:",len(ds))

Size of the dataset: 750


In [ ]:
#| eval: false
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


Next, the data loader definition and model loading:
- `ckpt_path`: Path to the model checkpoint

In [ ]:
#| eval: false
batch_size=8
data_loader = DataLoader(
    ds, 
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

ckpt_path = "/workspace/models/v3_20ep.pt"
model = torch.load(ckpt_path)

The following code snippet performs inference and stores the labels predicted by the model for each frame in a NumPy array.

In [ ]:
#| eval: false
def get_prediction(single_pred):
    np_pred = single_pred[0].detach().cpu().numpy()
    pred_labels = np.argmax(np_pred, axis=0)  
    return pred_labels

def inference(model, data_loader, device):
    model.eval()
    num_batches = 0
    pred_np = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Inference", leave=True):
            num_batches += 1

            test_item = {key: value.to(device) for key, value in batch.items()}
            img = test_item['frame']
            label = test_item['label']
            mask = test_item['mask']

            label[~mask] = 0
            pred = model(img) 
            for i in range(pred.shape[0]):
                single_pred = pred[i].unsqueeze(0)  # Shape: (1, C, H, W)
                pred_np.append(get_prediction(single_pred))
                
    return pred_np

pred_np = inference(model, data_loader, device)

Inference: 100%|██████████| 94/94 [00:03<00:00, 26.25it/s]


Next, an auxiliary array is created based on the dataset, aiming to replace all non-zero points (i.e., previously annotated points) with a value that distinguishes them from others during the verification process.

The chosen value was 99, which in the `semantic-kitti.yaml` file corresponds to the other-object class — a class that is not used in AIRCloud.

In [ ]:
#| eval: false
aux_np = []
for frame in ds:
    step_1 = np.where(frame["label"] == -1, 0, frame["label"])
    step_2 = np.where(step_1 != 0, 99, step_1)
    aux_np.append(step_2)
np.unique(aux_np)

array([ 0, 99], dtype=int32)

Finally, the `filtered_prediction` array is created. For each point equal to zero in the auxiliary array (i.e., points without prior annotation), the value is replaced by the model’s prediction. When the point is non-zero — specifically, 99 —, the value is preserved.

In this way, the final array contains the verified annotation value if it exists, and otherwise, it takes the predicted value.

In [ ]:
#| eval: false
filtered_prediction = [
    np.where(a == 0, p, a)   
    for a, p in zip(aux_np, pred_np)
]
np.unique(filtered_prediction[0])

array([ 1,  3,  4,  5, 99])

## Converting the results

With the prediction already filtered, the next step is to convert the model-predicted range images back into point clouds, so that the label verification process can be performed.

To achieve this, the following function performs the conversion using the z-buffer graphics strategy to handle occlusion issues that arise during the transition between different dimensions.

In [ ]:
#| eval: false
def image_to_cloud_zbuffer(predicted_labels, proj):
    idx_map = proj.inverse_projection["idx_map"]
    N       = proj.inverse_projection["proj_x"].shape[0]  

    recovered = np.zeros(N, dtype=predicted_labels.dtype)
    mask_pix   = idx_map >= 0
    point_ids  = idx_map[mask_pix]        
    pix_labels = predicted_labels[mask_pix]  
    recovered[point_ids] = pix_labels
    return recovered

recovered_labels = []
for label, data in zip(filtered_prediction, ds):
    rec = image_to_cloud_zbuffer(label, proj)
    recovered_labels.append(rec)

recovered_labels[0:5]

[array([ 0,  0,  0, ...,  0,  0, 99]),
 array([ 0,  0, 99, ...,  0,  0,  0]),
 array([ 0,  0,  0, ..., 99,  0,  0]),
 array([ 0, 99,  0, ..., 99,  0,  0]),
 array([ 0, 99,  0, ..., 99,  0,  0])]

Since AIRCloud contains points with NaN values due to the nature of LiDAR operation, and given the decision to handle this issue at runtime, removing these points during dataset loading — as AIRCloud itself does — changes the total number of points per frame. This leads to dimensional inconsistencies when compared with the original raw files.

To address this issue and preserve consistency in size, zeros (unlabeled) will be inserted at all positions where the raw LiDAR readings contain NaN values.

In [ ]:
#| eval: false
nan_dataset = SemanticKITTIDataset(data_path=DATASET_PATH, split='valid')
print("Size of test dataset: ", len(nan_dataset))

new_rec_labels = []
for i in range(len(recovered_labels)):
    aux = np.zeros(nan_dataset[i]["frame"].shape[0])
    valid_mask = ~np.isnan(nan_dataset[i]["frame"]).any(axis=1)
    aux[valid_mask] = recovered_labels[i]
    new_rec_labels.append(aux.astype(int))

Size of test dataset:  750


Thus, the number of points between the LiDAR readings and the labels is the same.

In [ ]:
#| eval: false
shape_problem = False
for i in range(len(new_rec_labels)):
    if nan_dataset[i]["frame"].shape[0] != new_rec_labels[i].shape[0]:
        shape_problem = True
        break

if shape_problem:
    print("ERROR: The target dataset and the generated labels have frames with different shapes")
else:
    print("The target dataset and the generated labels have shapes of the same size, proceeding.")

The target dataset and the generated labels have shapes of the same size, proceeding.


With the labels already restored to 3D space and with the shape corrected, the next step is to perform the inverse mapping of the labels. This is necessary because, during data loading with the SemanticKITTIDataset class, the labels are naturally mapped to indexes, a process required for providing these data to the models.

It is worth noting that the learning_map_inv from the dataset’s .yaml file should not be used, since the mapping applied here includes a specific handling for the other-object class, which will be used to mark previously verified labels.

In [ ]:
#| eval: false
inv_map = {
    0: 0,     # "unlabeled"
    1: 10,    # "car"
    2: 30,    # "person"
    3: 40,    # "road"
    4: 70,    # "vegetation"
    5: 72,    # "terrain"
    99: 99    # "other-object" to "other-object" 
}

mapped_labels = []

for i in range(len(new_rec_labels)):
    map_label = [inv_map[label_id] for label_id in new_rec_labels[i]]
    mapped_labels.append(np.array(map_label))

np.unique(mapped_labels[0])

array([ 0, 10, 40, 70, 72, 99])

## Saving and downloading

This cell converts the mapped labels to `uint32`, saves them temporarily, compresses all files into a `.zip` archive named after the sequence, and then removes the temporary folder used for storage. 

The generated zip will contain the labels in the correct format for use with the annotation tool — simply create a copy of the target sequence from the original dataset, replace the original labels with the predicted and formatted ones, and proceed with the correction process.


In [ ]:
#| eval: false
labels_path = os.path.join(ds.labels_path, SEQUENCE, "labels")
label_list = sorted(os.listdir(labels_path))

saving_folder = os.path.join(OUTPUT_PATH, "temp_labels/")
if not os.path.exists(saving_folder):
    os.makedirs(saving_folder)
    
for i in range(len(label_list)):
    final_labels_uint32 = mapped_labels[i].astype(np.uint32)
    with open(saving_folder + label_list[i], "wb") as f:
        final_labels_uint32.tofile(f)

zip_filename = os.path.join(OUTPUT_PATH, SEQUENCE)
shutil.make_archive(zip_filename, 'zip', saving_folder)
shutil.rmtree(saving_folder)

print(f"Labels ready for download at {zip_filename}.zip")

Labels ready for download at /workspace/converted_labels/08.zip
